In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score
import seaborn as sns
import keras_tuner as kt  # Import keras-tuner

# Load the data
train_df = pd.read_csv('bugs-train.csv')
test_df = pd.read_csv('bugs-test.csv')

# Define severity code mapping starting from 1
severity_mapping = {
    'enhancement': 1,
    'trivial': 2,
    'minor': 3,
    'normal': 4,
    'major': 5,
    'blocker': 6,
    'critical': 7
}

# Apply the mapping
train_df['severity'] = train_df['severity'].map(severity_mapping)

# Ensure all labels are within the valid range by adjusting them to start from 0
train_df['severity'] = train_df['severity'] - 1

# Print unique values to check the range
print("Unique severity values after mapping and adjustment:", train_df['severity'].unique())

# Load the BERT tokenizer
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Function to further split long words into subwords
def split_long_word(word, max_length=10):
    return [word[i:i + max_length] for i in range(0, len(word), max_length)]

# Function to tokenize text data into subwords with special handling for long words
def custom_subword_tokenize(text, max_word_length=25):
    tokens = []
    for word in text.split():
        if len(word) > max_word_length:
            # Split long words into smaller subwords
            subwords = split_long_word(word)
            for subword in subwords:
                tokens.extend(bert_tokenizer.tokenize(subword))
        else:
            tokens.extend(bert_tokenizer.tokenize(word))
    return tokens

# Apply custom subword tokenization
if 'summary' in train_df.columns:
    train_df['tokens'] = train_df['summary'].apply(custom_subword_tokenize)
else:
    print("Column 'summary' not found in DataFrame")

if 'summary' in test_df.columns:
    test_df['tokens'] = test_df['summary'].apply(custom_subword_tokenize)
else:
    print("Column 'summary' not found in DataFrame")

# Convert tokens to a format suitable for padding (list of lists)
train_token_lists = train_df['tokens'].tolist()
test_token_lists = test_df['tokens'].tolist()

# Build a tokenizer and fit on the token lists
text_tokenizer = tf.keras.preprocessing.text.Tokenizer()
text_tokenizer.fit_on_texts(train_token_lists)

# Convert tokens to sequences of integers
X_train = text_tokenizer.texts_to_sequences(train_token_lists)
X_test = text_tokenizer.texts_to_sequences(test_token_lists)

# Padding sequences to ensure uniform length
max_length = 100  # Maximum length of sequences
X_train = pad_sequences(X_train, maxlen=max_length, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=max_length, padding='post', truncating='post')

# Labels
y_train = train_df['severity']

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Define a function to build the model for hyperparameter tuning
def build_model(hp):
    model = Sequential()
    model.add(Embedding(input_dim=len(text_tokenizer.word_index) + 1,
                        output_dim=hp.Int('embedding_output_dim', min_value=32, max_value=128, step=32),
                        input_length=max_length))
    model.add(Conv1D(filters=hp.Int('conv_filters', min_value=32, max_value=128, step=32),
                     kernel_size=5,
                     activation='relu'))
    model.add(GlobalMaxPooling1D())
    model.add(Dense(units=hp.Int('dense_units', min_value=32, max_value=128, step=32), activation='relu'))
    model.add(Dropout(rate=hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)))
    model.add(Dense(7, activation='softmax'))  # 7 classes for severity
    model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'rmsprop']),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Hyperparameter tuning using Keras Tuner
tuner = kt.Hyperband(build_model,
                     objective='val_accuracy',
                     max_epochs=10,
                     factor=3,
                     directory='my_dir',
                     project_name='bug_classification')

# Perform hyperparameter search
tuner.search(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best hyperparameters: {best_hps.values}")

# Build and train the model with the best hyperparameters
model = tuner.hypermodel.build(best_hps)
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val), batch_size=32)

# Plot the training and validation loss and accuracy
plt.figure(figsize=(12, 4))

# Plot training & validation accuracy values
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.show()

# Evaluate the model on the validation set
val_predictions = model.predict(X_val)
val_predictions_classes = np.argmax(val_predictions, axis=1)

# Confusion matrix
cm = confusion_matrix(y_val, val_predictions_classes)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Classification report
report = classification_report(y_val, val_predictions_classes, target_names=severity_mapping.keys())
print(report)

# Calculate precision, recall, and F1-score
precision = precision_score(y_val, val_predictions_classes, average='macro')
recall = recall_score(y_val, val_predictions_classes, average='macro')
f1 = f1_score(y_val, val_predictions_classes, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

# Save the model
model.save('bug_classification_cnn_model.h5')

# Testing the Model on Test Data

# Load and preprocess test data (already done previously)
# Make predictions on test data
test_predictions = model.predict(X_test)
test_predictions_classes = np.argmax(test_predictions, axis=1)

# Save test predictions (only 'bug_id' and 'predicted_severity')
test_predictions_df = test_df[['bug_id']].copy()
test_predictions_df['predicted_severity'] = test_predictions_classes + 1  # Adjust back to original labels
test_predictions_df.to_csv('test_predictions.csv', index=False)

# Display the first few predictions
print(test_predictions_df.head(15))